# MVP v0.3.2.2: Guidance Hyperparameter Sweep

**Date:** 2026-03-12  
**Builds on:** v0.3.2.1 (shared diffuser)

## Motivation

v0.3.2.1 showed that guidance (action_scale=0.2, ratio=0.25) kills all successes,
dropping SR from 60% (unguided) to 0%. The negative guidance pushes away from
expert-dominated behavior data, which is counterproductive.

This notebook sweeps action_scale and ratio to find the right guidance regime.

## Sweep Grid

- `action_scale`: [0.0, 0.05, 0.1, 0.2]
- `ratio`: [0.0, 0.1, 0.25]

11 unique configs (action_scale=0 is the same regardless of ratio) × 4 policies = 44 evaluations

In [1]:
import sys, os
import importlib
import numpy as np
import torch
import torch.nn as nn
import h5py
import json
import math
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display
from pathlib import Path
from collections import OrderedDict
from copy import deepcopy
from tqdm import tqdm
from scipy import stats as sp_stats
import time

PROJECT_ROOT = Path("/home1/reishuen/latent_sope")
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "third_party" / "sope"))
sys.path.insert(0, str(PROJECT_ROOT / "third_party" / "robomimic"))

from opelab.core.baselines.diffusion.temporal import TemporalUnet
from opelab.core.baselines.diffusion.diffusion import GaussianDiffusion
from opelab.core.baselines.diffusion.helpers import EMA, apply_conditioning

import robomimic.utils.file_utils as FileUtils
import robomimic.utils.obs_utils as ObsUtils

import latent_sope.robomimic_interface.guidance as _guidance_mod
importlib.reload(_guidance_mod)
from latent_sope.robomimic_interface.checkpoints import (
    load_checkpoint, build_algo_from_checkpoint
)
from latent_sope.robomimic_interface.guidance import RobomimicDiffusionScorer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CLIPTextModelWithProjection LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...23}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.layer_norm1.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.out_proj.bias   | UNEXPECTED |  | 
vision_model.embeddings.position_ids                           | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.k_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.v_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.k_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.l

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Device: cuda


## Configuration

In [2]:
OBS_KEYS = ["object", "robot0_eef_pos", "robot0_eef_quat", "robot0_gripper_qpos"]
STATE_DIM = 19
ACTION_DIM = 7
TRANSITION_DIM = STATE_DIM + ACTION_DIM

CHUNK_SIZE = 4
STRIDE = 2
N_DIFFUSION_STEPS = 256
DIM_MULTS = (1, 4, 8)
BASE_DIM = 32
ACTION_WEIGHT = 5.0
PREDICT_EPSILON = False

TRAIN_EPOCHS = 25
BATCH_SIZE = 64
LR = 3e-4
GRAD_CLIP = 1.0

NUM_TARGET_ROLLOUTS = 20
NUM_SYNTHETIC_TRAJS = 20
T_GEN = 60

CUBE_Z_INDEX = 2
LIFT_THRESHOLD = 0.84

# Fixed guidance params
K_GUIDE = 1
NORMALIZE_GRAD = True
USE_ADAPTIVE = False
CLAMP = False
L_INF = 1.0

# Sweep grid
ACTION_SCALES = [0.0, 0.05, 0.1, 0.2]
RATIOS = [0.0, 0.1, 0.25]

BC_EPOCHS = 500
BC_BATCH = 256
BC_HIDDEN = 256

DEMO_HDF5 = PROJECT_ROOT / "third_party/robomimic/datasets/lift/ph/low_dim_v15.hdf5"
CKPT_BASE = PROJECT_ROOT / "third_party/robomimic/diffusion_policy_trained_models"
ROLLOUT_BASE = PROJECT_ROOT / "rollouts"
RESULTS_DIR = PROJECT_ROOT / "results/2026-03-12"

POLICIES = {
    "50demos_epoch10":  CKPT_BASE / "lift_diffusion_50demos"  / "20260311134204",
    "200demos_epoch10": CKPT_BASE / "lift_diffusion_200demos" / "20260311141036",
    "10demos_epoch20":  CKPT_BASE / "lift_diffusion_10demos"  / "20260311115828",
    "200demos_epoch40": CKPT_BASE / "lift_diffusion_200demos" / "20260311141036",
}

def get_epoch(key):
    return int(key.split("epoch")[1])

ORACLE_JSON = RESULTS_DIR / "oracle_eval_all_checkpoints.json"

# Build sweep configs (deduplicate action_scale=0)
sweep_configs = []
for scale in ACTION_SCALES:
    if scale == 0.0:
        sweep_configs.append((0.0, 0.0))
    else:
        for ratio in RATIOS:
            sweep_configs.append((scale, ratio))

print(f"Sweep: {len(sweep_configs)} configs x {len(POLICIES)} policies = {len(sweep_configs) * len(POLICIES)} evaluations")
for s, r in sweep_configs:
    print(f"  scale={s}, ratio={r}")

Sweep: 10 configs x 4 policies = 40 evaluations
  scale=0.0, ratio=0.0
  scale=0.05, ratio=0.0
  scale=0.05, ratio=0.1
  scale=0.05, ratio=0.25
  scale=0.1, ratio=0.0
  scale=0.1, ratio=0.1
  scale=0.1, ratio=0.25
  scale=0.2, ratio=0.0
  scale=0.2, ratio=0.1
  scale=0.2, ratio=0.25


## Load Oracle Values

In [3]:
with open(ORACLE_JSON, "r") as f:
    oracle_data = json.load(f)

oracle_values = {}
for key in POLICIES:
    od = oracle_data[key]
    oracle_values[key] = {
        "mean_return": od["mean_return"],
        "std_return": od["std_return"],
        "success_rate": od["success_rate"],
    }
    print(f"  {key:>20s}: V^pi={od['mean_return']:.4f}, SR={od['success_rate']*100:.1f}%")

       50demos_epoch10: V^pi=0.0000, SR=0.0%
      200demos_epoch10: V^pi=0.1800, SR=18.0%
       10demos_epoch20: V^pi=0.5200, SR=52.0%
      200demos_epoch40: V^pi=0.9000, SR=90.0%


## Load Data, Train Diffuser & BC (same as v0.3.2.1)

In [4]:
all_episodes = []
all_states_list = []
all_actions_list = []

# Expert demos
with h5py.File(DEMO_HDF5, "r") as f:
    demo_keys = sorted(f["data"].keys(), key=lambda x: int(x.split("_")[1]))
    for dk in tqdm(demo_keys, desc="Loading expert demos"):
        demo = f[f"data/{dk}"]
        obs_arrays = [demo["obs"][k][:].astype(np.float32) for k in OBS_KEYS]
        states = np.concatenate(obs_arrays, axis=-1)
        actions = demo["actions"][:].astype(np.float32)
        rewards = demo["rewards"][:].astype(np.float32)
        episode = {
            "states": states[:-1], "actions": actions[:-1],
            "rewards": rewards[:-1], "next-states": states[1:],
        }
        all_episodes.append(episode)
        all_states_list.append(states)
        all_actions_list.append(actions)

n_expert = len(all_episodes)
print(f"Loaded {n_expert} expert demos")

# Target rollouts
target_data_by_policy = {}
for policy_key in POLICIES:
    rollout_dir = ROLLOUT_BASE / f"multi_policy_{policy_key}"
    existing = sorted(rollout_dir.glob("rollout_*.h5"))
    assert len(existing) >= NUM_TARGET_ROLLOUTS
    policy_episodes = []
    for path in existing[:NUM_TARGET_ROLLOUTS]:
        with h5py.File(path, "r") as f:
            latents = f["latents"][:]
            actions = f["actions"][:]
            rewards = f["rewards"][:] if "rewards" in f else np.zeros(len(actions))
        states = latents[:, -1, :] if latents.ndim == 3 else latents
        states = states.astype(np.float32)
        actions = actions.astype(np.float32)
        episode = {
            "states": states[:-1], "actions": actions[:-1],
            "rewards": rewards[:-1] if len(rewards) > 1 else rewards,
            "next-states": states[1:],
        }
        policy_episodes.append(episode)
        all_episodes.append(episode)
        all_states_list.append(states)
        all_actions_list.append(actions)
    target_data_by_policy[policy_key] = policy_episodes
    env_sr = np.mean([ep["rewards"].sum() > 0 for ep in policy_episodes])
    print(f"  {policy_key:>20s}: {len(policy_episodes)} rollouts, env SR={env_sr*100:.0f}%")

n_target = len(all_episodes) - n_expert
print(f"\nTotal: {len(all_episodes)} episodes ({n_expert} expert + {n_target} target)")

Loading expert demos:   0%|          | 0/200 [00:00<?, ?it/s]

Loading expert demos:  65%|██████▌   | 130/200 [00:00<00:00, 1290.18it/s]

Loading expert demos: 100%|██████████| 200/200 [00:00<00:00, 1282.54it/s]

Loaded 200 expert demos
       50demos_epoch10: 20 rollouts, env SR=0%


      200demos_epoch10: 20 rollouts, env SR=0%
       10demos_epoch20: 20 rollouts, env SR=0%
      200demos_epoch40: 20 rollouts, env SR=0%

Total: 280 episodes (200 expert + 80 target)


In [5]:
# Normalization & chunking
all_states_cat = np.concatenate(all_states_list, axis=0)
all_actions_cat = np.concatenate(all_actions_list, axis=0)
norm_mean = np.concatenate([all_states_cat.mean(0), all_actions_cat.mean(0)]).astype(np.float32)
norm_std = np.maximum(np.concatenate([all_states_cat.std(0), all_actions_cat.std(0)]), 1e-6).astype(np.float32)
norm_mean_t = torch.tensor(norm_mean, device=device)
norm_std_t = torch.tensor(norm_std, device=device)
normalize_fn = lambda x, m=norm_mean_t, s=norm_std_t: (x - m) / s
unnormalize_fn = lambda x, m=norm_mean_t, s=norm_std_t: x * s + m

chunks_s, chunks_a = [], []
for ep in all_episodes:
    st, act = ep["states"], ep["actions"]
    for t0 in range(0, len(st) - CHUNK_SIZE, STRIDE):
        chunks_s.append(st[t0:t0+CHUNK_SIZE+1])
        chunks_a.append(act[t0:t0+CHUNK_SIZE])
chunks_s = np.array(chunks_s, dtype=np.float32)
chunks_a = np.array(chunks_a, dtype=np.float32)
train_x = np.concatenate([chunks_s[:, :-1, :], chunks_a], axis=-1)
train_cond = chunks_s[:, 0, :]
train_x_t = torch.tensor(train_x, dtype=torch.float32, device=device)
train_cond_t = torch.tensor(train_cond, dtype=torch.float32, device=device)
print(f"Chunks: {len(train_x_t)}, batches/epoch: {len(train_x_t) // BATCH_SIZE}")

Chunks: 6421, batches/epoch: 100


In [6]:
# Train diffuser
temporal_model = TemporalUnet(
    horizon=CHUNK_SIZE, transition_dim=TRANSITION_DIM,
    dim=BASE_DIM, dim_mults=DIM_MULTS, attention=False,
).to(device)
diffusion_model = GaussianDiffusion(
    model=temporal_model, horizon=CHUNK_SIZE,
    observation_dim=STATE_DIM, action_dim=ACTION_DIM,
    n_timesteps=N_DIFFUSION_STEPS,
    normalizer=normalize_fn, unnormalizer=unnormalize_fn,
    predict_epsilon=PREDICT_EPSILON, loss_type="l2",
    clip_denoised=False, action_weight=ACTION_WEIGHT,
    loss_weights=None, loss_discount=1.0,
).to(device)
ema = EMA(diffusion_model)
optimizer = torch.optim.Adam(diffusion_model.parameters(), lr=LR)

t0_train = time.time()
diffusion_model.train()
epoch_losses = []
for ep_num in range(1, TRAIN_EPOCHS + 1):
    perm = torch.randperm(len(train_x_t), device=device)
    ep_loss = []
    for start in range(0, len(train_x_t) - BATCH_SIZE + 1, BATCH_SIZE):
        idx = perm[start:start+BATCH_SIZE]
        x_norm = normalize_fn(train_x_t[idx])
        c_norm = normalize_fn(
            torch.cat([train_cond_t[idx], torch.zeros(BATCH_SIZE, ACTION_DIM, device=device)], dim=-1)
        )[:, :STATE_DIM]
        loss, _ = diffusion_model.loss(x_norm, {0: c_norm})
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(diffusion_model.parameters(), GRAD_CLIP)
        optimizer.step()
        ema.update(diffusion_model)
        ep_loss.append(loss.item())
    epoch_losses.append(np.mean(ep_loss))
    if ep_num % 5 == 0 or ep_num == 1:
        print(f"  Epoch {ep_num:>3d}/{TRAIN_EPOCHS}: loss={epoch_losses[-1]:.6f}")

elapsed_train = time.time() - t0_train
print(f"Diffuser trained: {elapsed_train:.0f}s, final loss={epoch_losses[-1]:.6f}")

[ models/temporal ] Channel dimensions: [(26, 32), (32, 128), (128, 256)]
[(26, 32), (32, 128), (128, 256)]


/home1/reishuen/latent_sope/third_party/sope/opelab/core/baselines/diffusion/diffusion.py:314: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  betas * np.sqrt(alphas_cumprod_prev) / (1. - alphas_cumprod))


  Epoch   1/25: loss=0.722507


  Epoch   5/25: loss=0.392404


  Epoch  10/25: loss=0.273080


  Epoch  15/25: loss=0.212237


  Epoch  20/25: loss=0.184410


  Epoch  25/25: loss=0.161790
Diffuser trained: 178s, final loss=0.161790


In [7]:
# Helper functions
class BCGaussian(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
        )
        self.mean_head = nn.Linear(hidden_dim, action_dim)
        self.log_std_head = nn.Linear(hidden_dim, action_dim)
    def forward(self, state):
        h = self.net(state)
        return self.mean_head(h), self.log_std_head(h).clamp(-5, 2)
    def log_prob(self, state, action):
        mean, log_std = self.forward(state)
        std = torch.exp(log_std)
        return -0.5 * (((action - mean) / std) ** 2 + 2 * log_std + math.log(2 * math.pi)).sum(dim=-1)
    def grad_log_prob(self, state, action):
        with torch.no_grad():
            mean, log_std = self.forward(state)
            std = torch.exp(log_std)
            return -(action - mean) / (std ** 2)
    def grad_log_prob_chunk(self, states, actions):
        B, T, _ = states.shape
        return self.grad_log_prob(
            states.reshape(B*T, -1), actions.reshape(B*T, -1)
        ).reshape(B, T, -1)


def get_initial_states_from_data(data, num_samples, device):
    all_initial = np.array([ep["states"][0] for ep in data])
    indices = np.random.choice(len(all_initial), num_samples, replace=True)
    return torch.tensor(all_initial[indices], dtype=torch.float32, device=device)


def score_trajectories_gt(trajectories, cube_z_index, threshold):
    cube_z = trajectories[:, :, cube_z_index]
    successes = np.any(cube_z > threshold, axis=1)
    return successes.astype(np.float32), successes


def generate_trajectories_full_guidance(
    diffusion_model, initial_states,
    normalize_fn, unnormalize_fn,
    state_dim, action_dim,
    chunk_size, t_gen, device,
    target_scorer=None, behavior_scorer=None,
    action_scale=0.0, ratio=0.5,
    k_guide=1, normalize_grad=True,
    use_adaptive=False, clamp=False, l_inf=1.0,
):
    guided = (target_scorer is not None and action_scale > 0)
    use_neg_grad = (behavior_scorer is not None and ratio > 0)
    batch_size = initial_states.shape[0]
    transition_dim = state_dim + action_dim
    n_timesteps = diffusion_model.n_timesteps

    padded = torch.cat([initial_states, torch.zeros(batch_size, action_dim, device=device)], dim=1)
    normalized_initial = normalize_fn(padded)[:, :state_dim]
    all_trajectories = torch.zeros(batch_size, t_gen, transition_dim, device=device)
    conditions = {0: normalized_initial}
    total_generated = 0

    while total_generated < t_gen:
        shape = (batch_size, chunk_size, transition_dim)
        x = torch.randn(shape, device=device)
        x = apply_conditioning(x, conditions, state_dim)

        for t_diff in reversed(range(n_timesteps)):
            t_tensor = torch.full((batch_size,), t_diff, device=device, dtype=torch.long)
            with torch.no_grad():
                model_mean, _, model_log_variance = diffusion_model.p_mean_variance(x=x, t=t_tensor)
                model_std = torch.exp(0.5 * model_log_variance)

            if guided:
                model_mean = unnormalize_fn(model_mean)
                for _k in range(k_guide):
                    states_chunk = model_mean[:, :, :state_dim]
                    actions_chunk = model_mean[:, :, state_dim:]
                    target_grad = target_scorer.grad_log_prob_chunk(states_chunk, actions_chunk)
                    if use_neg_grad:
                        behavior_grad = behavior_scorer.grad_log_prob_chunk(states_chunk, actions_chunk)
                    if normalize_grad:
                        eps = 1e-6
                        target_grad = target_grad / (target_grad.norm(dim=-1, keepdim=True) + eps)
                        if use_neg_grad:
                            behavior_grad = behavior_grad / (behavior_grad.norm(dim=-1, keepdim=True) + eps)
                    if clamp:
                        target_grad = target_grad.clamp(-l_inf, l_inf)
                        if use_neg_grad:
                            behavior_grad = behavior_grad.clamp(-l_inf, l_inf)
                    if use_neg_grad:
                        guide_actions = target_grad - ratio * behavior_grad
                    else:
                        guide_actions = target_grad
                    guide = torch.zeros_like(model_mean)
                    guide[:, :, state_dim:] = guide_actions
                    if use_adaptive:
                        scale_mult = 0.5 * (1 + math.cos(math.pi * (n_timesteps - t_diff) / n_timesteps))
                        guide = scale_mult * action_scale * guide
                    else:
                        guide = action_scale * guide
                    model_mean = model_mean + guide
                    model_mean = normalize_fn(model_mean)
                    model_mean = apply_conditioning(model_mean, conditions, state_dim)
                    model_mean = unnormalize_fn(model_mean)
                model_mean = normalize_fn(model_mean)

            noise = torch.randn_like(x)
            nonzero_mask = (1 - (t_diff == 0) * 1.0)
            x = model_mean + nonzero_mask * model_std * noise
            x = apply_conditioning(x, conditions, state_dim)

        chunk_unnorm = unnormalize_fn(x)
        steps_to_store = min(chunk_size - 1, t_gen - total_generated)
        all_trajectories[:, total_generated:total_generated + steps_to_store] = chunk_unnorm[:, :steps_to_store]
        total_generated += steps_to_store
        if total_generated >= t_gen:
            break
        conditions = {0: x[:, -1, :state_dim]}

    return all_trajectories.detach().cpu().numpy()


print("Helper functions defined.")

Helper functions defined.


In [8]:
# Train BC
bc_states = np.concatenate([ep["states"] for ep in all_episodes], axis=0)
bc_actions = np.concatenate([ep["actions"] for ep in all_episodes], axis=0)
bc_behavior = BCGaussian(STATE_DIM, ACTION_DIM, hidden_dim=BC_HIDDEN).to(device)
bc_optimizer = torch.optim.Adam(bc_behavior.parameters(), lr=1e-3)
states_t = torch.tensor(bc_states, dtype=torch.float32, device=device)
actions_t = torch.tensor(bc_actions, dtype=torch.float32, device=device)
bc_behavior.train()
for bc_ep in range(BC_EPOCHS):
    idx = torch.randint(0, len(states_t), (BC_BATCH,), device=device)
    nll = -bc_behavior.log_prob(states_t[idx], actions_t[idx]).mean()
    bc_optimizer.zero_grad()
    nll.backward()
    bc_optimizer.step()
bc_behavior.eval()
print(f"BC trained: NLL={nll.item():.4f}")

BC trained: NLL=-6.2461


## Sweep: Evaluate all (scale, ratio) configs across all policies

In [9]:
# Results: sweep_results[(scale, ratio)][policy_key] = {ope, sr, ...}
sweep_results = {}
t0_sweep = time.time()
total_evals = len(sweep_configs) * len(POLICIES)
eval_count = 0

# Pre-load target scorers (reuse across configs)
target_scorers = {}
for policy_key, run_dir in POLICIES.items():
    epoch = get_epoch(policy_key)
    ckpt = load_checkpoint(str(run_dir), ckpt_path=f"models/model_epoch_{epoch}.pth")
    target_algo = build_algo_from_checkpoint(ckpt, device=str(device))
    target_scorers[policy_key] = RobomimicDiffusionScorer(target_algo, device=str(device), obs_keys=OBS_KEYS)
print(f"Loaded {len(target_scorers)} target policy scorers")

for cfg_idx, (scale, ratio) in enumerate(sweep_configs):
    cfg_key = (scale, ratio)
    sweep_results[cfg_key] = {}
    
    for policy_key in POLICIES:
        eval_count += 1
        oracle_v = oracle_values[policy_key]["mean_return"]
        
        np.random.seed(42 + list(POLICIES.keys()).index(policy_key))
        torch.manual_seed(42 + list(POLICIES.keys()).index(policy_key))
        
        initial_states = get_initial_states_from_data(
            target_data_by_policy[policy_key], NUM_SYNTHETIC_TRAJS, device
        )
        
        trajs = generate_trajectories_full_guidance(
            diffusion_model=ema.ema_model,
            initial_states=initial_states,
            normalize_fn=normalize_fn, unnormalize_fn=unnormalize_fn,
            state_dim=STATE_DIM, action_dim=ACTION_DIM,
            chunk_size=CHUNK_SIZE, t_gen=T_GEN, device=device,
            target_scorer=target_scorers[policy_key] if scale > 0 else None,
            behavior_scorer=bc_behavior if (scale > 0 and ratio > 0) else None,
            action_scale=scale, ratio=ratio,
            k_guide=K_GUIDE, normalize_grad=NORMALIZE_GRAD,
            use_adaptive=USE_ADAPTIVE, clamp=CLAMP, l_inf=L_INF,
        )
        
        returns, successes = score_trajectories_gt(trajs, CUBE_Z_INDEX, LIFT_THRESHOLD)
        ope = float(np.mean(returns))
        sr = float(np.mean(successes))
        
        sweep_results[cfg_key][policy_key] = {
            "ope": ope, "sr": sr, "oracle": oracle_v,
            "rel_error": abs(ope - oracle_v) / max(abs(oracle_v), 1e-6),
        }
    
    # Print config summary
    opes = [sweep_results[cfg_key][k]["ope"] for k in POLICIES]
    oracles = [sweep_results[cfg_key][k]["oracle"] for k in POLICIES]
    if len(set(opes)) > 1:
        rho, _ = sp_stats.spearmanr(oracles, opes)
    else:
        rho = float('nan')
    srs = [sweep_results[cfg_key][k]["sr"] for k in POLICIES]
    print(f"[{eval_count}/{total_evals}] scale={scale}, ratio={ratio}: "
          f"SRs={[f'{s*100:.0f}%' for s in srs]}, rho={rho:.3f}")

elapsed_sweep = time.time() - t0_sweep
print(f"\nSweep complete: {elapsed_sweep/60:.1f} min")

# Clean up scorers
del target_scorers
torch.cuda.empty_cache()


============= Initialized Observation Utils with Obs Spec =============

using obs modality: low_dim with keys: ['robot0_eef_pos', 'robot0_eef_quat', 'robot0_gripper_qpos', 'object']
using obs modality: rgb with keys: []
using obs modality: depth with keys: []
using obs modality: scan with keys: []


number of parameters: 6.576359e+07


[18:13:32] INFO     build_algo_from_checkpoint took 0.27 seconds to execute                           ]8;id=295826;file:///home1/reishuen/latent_sope/src/latent_sope/utils/common.py\common.py]8;;\:]8;id=716246;file:///home1/reishuen/latent_sope/src/latent_sope/utils/common.py#137\137]8;;\


============= Initialized Observation Utils with Obs Spec =============

using obs modality: low_dim with keys: ['robot0_eef_pos', 'robot0_eef_quat', 'robot0_gripper_qpos', 'object']
using obs modality: rgb with keys: []
using obs modality: depth with keys: []
using obs modality: scan with keys: []
number of parameters: 6.576359e+07


[18:13:33] INFO     build_algo_from_checkpoint took 0.20 seconds to execute                           ]8;id=136237;file:///home1/reishuen/latent_sope/src/latent_sope/utils/common.py\common.py]8;;\:]8;id=477553;file:///home1/reishuen/latent_sope/src/latent_sope/utils/common.py#137\137]8;;\

...

[24/40] scale=0.1, ratio=0.1: SRs=['5%', '0%', '0%', '0%'], rho=-0.775


[28/40] scale=0.1, ratio=0.25: SRs=['5%', '0%', '0%', '5%'], rho=0.000


[32/40] scale=0.2, ratio=0.0: SRs=['0%', '0%', '0%', '0%'], rho=nan


[36/40] scale=0.2, ratio=0.1: SRs=['0%', '0%', '5%', '0%'], rho=0.258


[40/40] scale=0.2, ratio=0.25: SRs=['10%', '0%', '5%', '0%'], rho=-0.632

Sweep complete: 26.4 min


## Results: Summary Table

In [10]:
sorted_policy_keys = sorted(POLICIES.keys(), key=lambda k: oracle_values[k]["mean_return"])

# Header
header = f"{'scale':>6s} {'ratio':>6s} | "
header += " | ".join([f"{k[:12]:>12s}" for k in sorted_policy_keys])
header += " | {'rho':>6s} {'MAE':>6s}"
print(header)
print("-" * len(header))

for (scale, ratio), results in sweep_results.items():
    oracles = [results[k]["oracle"] for k in sorted_policy_keys]
    opes = [results[k]["ope"] for k in sorted_policy_keys]
    srs = [results[k]["sr"] for k in sorted_policy_keys]
    
    if len(set(opes)) > 1:
        rho, _ = sp_stats.spearmanr(oracles, opes)
    else:
        rho = float('nan')
    mae = np.mean(np.abs(np.array(oracles) - np.array(opes)))
    
    row = f"{scale:>6.2f} {ratio:>6.2f} | "
    row += " | ".join([f"{ope:.2f} ({sr*100:3.0f}%)" for ope, sr in zip(opes, srs)])
    row += f" | {rho:>6.3f} {mae:>6.3f}"
    print(row)

# Print oracle values for reference
print()
row = f"{'ORACLE':>13s} | "
row += " | ".join([f"{oracle_values[k]['mean_return']:.2f} ({oracle_values[k]['success_rate']*100:3.0f}%)" for k in sorted_policy_keys])
print(row)

 scale  ratio | 50demos_epoc | 200demos_epo | 10demos_epoc | 200demos_epo | {'rho':>6s} {'MAE':>6s}
---------------------------------------------------------------------------------------------------
  0.00   0.00 | 0.30 ( 30%) | 0.25 ( 25%) | 0.25 ( 25%) | 0.10 ( 10%) | -0.949  0.360
  0.05   0.00 | 0.00 (  0%) | 0.10 ( 10%) | 0.00 (  0%) | 0.00 (  0%) | -0.258  0.375
  0.05   0.10 | 0.00 (  0%) | 0.00 (  0%) | 0.00 (  0%) | 0.00 (  0%) |    nan  0.400
  0.05   0.25 | 0.00 (  0%) | 0.00 (  0%) | 0.00 (  0%) | 0.05 (  5%) |  0.775  0.387
  0.10   0.00 | 0.00 (  0%) | 0.05 (  5%) | 0.00 (  0%) | 0.00 (  0%) | -0.258  0.387
  0.10   0.10 | 0.05 (  5%) | 0.00 (  0%) | 0.00 (  0%) | 0.00 (  0%) | -0.775  0.413
  0.10   0.25 | 0.05 (  5%) | 0.00 (  0%) | 0.00 (  0%) | 0.05 (  5%) |  0.000  0.400
  0.20   0.00 | 0.00 (  0%) | 0.00 (  0%) | 0.00 (  0%) | 0.00 (  0%) |    nan  0.400
  0.20   0.10 | 0.00 (  0%) | 0.00 (  0%) | 0.05 (  5%) | 0.00 (  0%) |  0.258  0.387
  0.20   0.25 | 0.10 ( 10%

## Visualizations

In [11]:
# ── Figure 1: Heatmap of Spearman rho across sweep grid ──
unique_scales = sorted(set(s for s, r in sweep_configs if s > 0))
unique_ratios = sorted(set(r for s, r in sweep_configs if s > 0))

rho_matrix = np.full((len(unique_scales), len(unique_ratios)), np.nan)
mae_matrix = np.full((len(unique_scales), len(unique_ratios)), np.nan)

for i, scale in enumerate(unique_scales):
    for j, ratio in enumerate(unique_ratios):
        cfg_key = (scale, ratio)
        if cfg_key in sweep_results:
            oracles = [sweep_results[cfg_key][k]["oracle"] for k in POLICIES]
            opes = [sweep_results[cfg_key][k]["ope"] for k in POLICIES]
            if len(set(opes)) > 1:
                rho_matrix[i, j], _ = sp_stats.spearmanr(oracles, opes)
            mae_matrix[i, j] = np.mean(np.abs(np.array(oracles) - np.array(opes)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Rho heatmap
ax = axes[0]
im = ax.imshow(rho_matrix, cmap="RdYlGn", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(unique_ratios)))
ax.set_xticklabels([f"{r}" for r in unique_ratios])
ax.set_yticks(range(len(unique_scales)))
ax.set_yticklabels([f"{s}" for s in unique_scales])
ax.set_xlabel("ratio (negative guidance)")
ax.set_ylabel("action_scale")
ax.set_title("Spearman rho (higher = better ranking)")
for i in range(len(unique_scales)):
    for j in range(len(unique_ratios)):
        val = rho_matrix[i, j]
        txt = f"{val:.2f}" if not np.isnan(val) else "NaN"
        ax.text(j, i, txt, ha="center", va="center", fontsize=10, fontweight="bold")
fig.colorbar(im, ax=ax)

# MAE heatmap
ax = axes[1]
im = ax.imshow(mae_matrix, cmap="RdYlGn_r", vmin=0, vmax=0.5, aspect="auto")
ax.set_xticks(range(len(unique_ratios)))
ax.set_xticklabels([f"{r}" for r in unique_ratios])
ax.set_yticks(range(len(unique_scales)))
ax.set_yticklabels([f"{s}" for s in unique_scales])
ax.set_xlabel("ratio (negative guidance)")
ax.set_ylabel("action_scale")
ax.set_title("MAE (lower = better calibration)")
for i in range(len(unique_scales)):
    for j in range(len(unique_ratios)):
        val = mae_matrix[i, j]
        txt = f"{val:.3f}" if not np.isnan(val) else "NaN"
        ax.text(j, i, txt, ha="center", va="center", fontsize=10, fontweight="bold")
fig.colorbar(im, ax=ax)

plt.suptitle("v0.3.2.2: Guidance Sweep (Spearman rho & MAE)", fontweight="bold")
plt.tight_layout()
display(fig)
plt.close(fig)

<Figure size 1400x500 with 4 Axes>

In [12]:
# ── Figure 2: Per-policy SR across configs ──
fig, axes = plt.subplots(1, len(sorted_policy_keys), figsize=(5 * len(sorted_policy_keys), 4), sharey=True)

for ax, policy_key in zip(axes, sorted_policy_keys):
    oracle_sr = oracle_values[policy_key]["success_rate"]
    
    config_labels = []
    config_srs = []
    for (scale, ratio) in sweep_configs:
        label = f"s{scale}_r{ratio}"
        sr = sweep_results[(scale, ratio)][policy_key]["sr"]
        config_labels.append(label)
        config_srs.append(sr * 100)
    
    colors = ["steelblue" if s == 0 else "coral" for s, _ in sweep_configs]
    ax.bar(range(len(config_labels)), config_srs, color=colors)
    ax.axhline(y=oracle_sr * 100, color="green", linestyle="--", linewidth=1.5, label=f"Oracle SR={oracle_sr*100:.0f}%")
    ax.set_xticks(range(len(config_labels)))
    ax.set_xticklabels(config_labels, rotation=70, ha="right", fontsize=7)
    ax.set_title(f"{policy_key}\n(Oracle={oracle_sr*100:.0f}%)", fontsize=9)
    ax.set_ylim(0, 105)
    if ax == axes[0]:
        ax.set_ylabel("Synthetic SR (%)")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.2, axis="y")

plt.suptitle("v0.3.2.2: Synthetic SR by Config and Policy", fontweight="bold")
plt.tight_layout()
display(fig)
plt.close(fig)

<Figure size 2000x400 with 4 Axes>

In [13]:
# ── Figure 3: Oracle vs OPE for best config ──
# Find best config by Spearman rho (break ties by MAE)
best_cfg = None
best_rho = -2
best_mae = 999

for cfg_key in sweep_results:
    oracles = [sweep_results[cfg_key][k]["oracle"] for k in POLICIES]
    opes = [sweep_results[cfg_key][k]["ope"] for k in POLICIES]
    if len(set(opes)) > 1:
        rho, _ = sp_stats.spearmanr(oracles, opes)
    else:
        rho = float('nan')
    mae = np.mean(np.abs(np.array(oracles) - np.array(opes)))
    
    if not np.isnan(rho) and (rho > best_rho or (rho == best_rho and mae < best_mae)):
        best_rho = rho
        best_mae = mae
        best_cfg = cfg_key

print(f"Best config: scale={best_cfg[0]}, ratio={best_cfg[1]} (rho={best_rho:.3f}, MAE={best_mae:.3f})")

fig, ax = plt.subplots(1, 1, figsize=(6, 6))
for k in POLICIES:
    r = sweep_results[best_cfg][k]
    ax.scatter(r["oracle"], r["ope"], s=80, zorder=5)
    ax.annotate(k, (r["oracle"], r["ope"]), textcoords="offset points", xytext=(5, 5), fontsize=8)

ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="ideal")
ax.set_xlabel("Oracle V^pi")
ax.set_ylabel("OPE Estimate")
ax.set_title(f"Best Config: scale={best_cfg[0]}, ratio={best_cfg[1]}\nrho={best_rho:.3f}, MAE={best_mae:.3f}")
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.set_aspect("equal")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
display(fig)
plt.close(fig)

Best config: scale=0.05, ratio=0.25 (rho=0.775, MAE=0.387)


<Figure size 600x600 with 1 Axes>

## Summary

In [14]:
print(f"{'='*70}")
print("MVP v0.3.2.2 COMPLETE — Guidance Sweep")
print(f"{'='*70}")
print(f"Configs tested: {len(sweep_configs)}")
print(f"Total evaluations: {len(sweep_configs) * len(POLICIES)}")
print(f"Sweep time: {elapsed_sweep/60:.1f} min")
print(f"Training time: diffuser={elapsed_train:.0f}s")
print()
print(f"Best config: scale={best_cfg[0]}, ratio={best_cfg[1]}")
print(f"  Spearman rho: {best_rho:.3f}")
print(f"  MAE: {best_mae:.3f}")
print()
print("Per-policy results (best config):")
for k in sorted_policy_keys:
    r = sweep_results[best_cfg][k]
    print(f"  {k:>20s}: Oracle={r['oracle']:.2f}, OPE={r['ope']:.2f}, SR={r['sr']*100:.0f}%")

MVP v0.3.2.2 COMPLETE — Guidance Sweep
Configs tested: 10
Total evaluations: 40
Sweep time: 26.4 min
Training time: diffuser=178s

Best config: scale=0.05, ratio=0.25
  Spearman rho: 0.775
  MAE: 0.387

Per-policy results (best config):
       50demos_epoch10: Oracle=0.00, OPE=0.00, SR=0%
      200demos_epoch10: Oracle=0.18, OPE=0.00, SR=0%
       10demos_epoch20: Oracle=0.52, OPE=0.00, SR=0%
      200demos_epoch40: Oracle=0.90, OPE=0.05, SR=5%
